# User Configuration

In [ ]:
# Set these paths before running the notebook

# Encoding model results directory
results_indir = ""

# Path to group-level significance results pickle (used to restrict the searchlight)
group_res_path = ""

# Path to RSA model RDM directory
rsa_model_rdm_dir = ""

# Path to MNI template anatomy image
anat_img_path = ""



# fMRI data directory
fmri_dir = ""

# Annotation directories
hands_annotations_dir = ""
cvat_annotations_dir = ""

### Define parameters

In [ ]:
subjects = ['sub01', 'sub02','sub03','sub04','sub05','sub06']
template_subj = 'MNI152_2009_template_SSW'
outpath_template = 'subjects_results/{}_{}_results.p'


# ****************************
#   SEARCHLIGHT PARAMS
# ****************************
radius = 5 # in voxels
threshold = 0.5 # fraction of voxels needed in a searchlight's center+neighbors to keep it (meaning proportion of voxles who are not 0s/NaN)


### Define viz parameters

In [ ]:
import matplotlib as mpl
mpl.rcParams['font.family'] = 'Arial'


# While this can be done during the creation of all_rdms, doing so here allows flexibility if variations of the affordance models are added etc..
# Also allows flexible renaming with spaces, capitalization, etc.
clean_modelnames_dictmap = {
    'semantic_BGE' : 'Semantic',
    'visual' : 'Visual',
    'roleagnostic_affordance' : 'Action Affordances',
    'hands_linearcomb_affordance': 'Hand Posture Affordances'
    }

clean_condition_names = {'target' : 'Target Objects',
                            'object': 'Passive Objects'}

top_n = 10 # how many ROIs to plot in the barplor of mean per ROI



# quickflat params
quickflat_defaults = dict(roi_list=['Cortices'],
                          linecolor=(0.25,0.25,0.25),linewidth=2, labelsize=0,
                          with_colorbar=False
                          )


# mosaic plots
def cleanup_cbar(mosaic_plot):
    # ------- Only keep min/max values of cbar -----

    cbar = mosaic_plot._cbar

    # Get all tick labels
    ticklabels = cbar.ax.get_yticklabels()

    # Hide all but first and last
    for lab in ticklabels[1:-1]:
        lab.set_visible(False)


def custom_crosshair(nilearn_plot_img):
    '''
    Note: need to set draw_cross to False when calling plot_img for this to work
    '''
    nilearn_plot_img.draw_cross(color='black', alpha=0.5, linewidth=1)


### Imports

In [ ]:
import os, sys

# Resolve utility paths relative to this notebook's directory
_nb_dir = os.path.abspath('')
sys.path.append(os.path.join(_nb_dir, 'utils', 'fmri'))
sys.path.append(os.path.join(_nb_dir, 'utils', 'fmri', 'speechmodeltutorial'))
sys.path.append(os.path.join(_nb_dir, 'utils', 'fmri', 'utils_natcook'))

import os, sys, pickle, ast
from os.path import join
from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


from nilearn import image
from nilearn.masking import unmask
from nilearn.plotting import plot_img, plot_stat_map


import cortex

from scipy.spatial import distance
from scipy.spatial.distance import squareform



from statsmodels.stats.multitest import multipletests

# searchlight relevant pacakges
from rsatoolbox.util.searchlight import get_volume_searchlight, get_searchlight_RDMs, evaluate_models_searchlight
from rsatoolbox.model import ModelFixed
from rsatoolbox.inference import eval_fixed

# ======================================= Custom imports =============================================

# ----- import local functions
# -- Speech model tutorial functions

# -- Local fmri-general functions
from banded_ridge_reconstruct_fir_weights import banded_ridge_reconstruct_fir_weights


# -- Natcook-specific helpers
from FMRIPathConfig import FMRIPathConfig
from analyses.pycortex_utils import pycortex_plot_flatmap
from analyses.nilearn_utils import replace_ns_with_nan, average_list_of_imgs
from analyses.group_statistics import group_level_1sample_ttest


# ======================================= Input directories =============================================

# --- Define directories and paths
path_config = FMRIPathConfig(fmri_dir)
display('Path templates:', path_config.patterns)

# ======================================= Fixed parameters =============================================
# --- Other
n_jobs = -1
np.random.seed(42)


### Helper functions

In [ ]:
def drop_subcortical_rois(sorted_labels, sorted_means, sorted_stds, subcortical_rois=['Declive']):
    filtered_labels = []
    filtered_means = []
    filtered_stds = []
    dropped_labels = []

    for label, mean, std in zip(sorted_labels, sorted_means, sorted_stds):
        # case-insensitive substring match
        if any(sub_roi.lower() in label.lower() for sub_roi in subcortical_rois):
            dropped_labels.append(label)
            continue

        filtered_labels.append(label)
        filtered_means.append(mean)
        filtered_stds.append(std)

    print(f'Dropped subcortical ROIs: {dropped_labels}')

    return filtered_labels, filtered_means, filtered_stds

### Create the model RDMs

Helper functions


In [ ]:
def read_embs_and_make_rdms(embs_path, exemplars):
    with open(embs_path, 'rb') as f:
        embs_dict = pickle.load(f)
    
    # Check which exemplars are actually present
    available_exemplars = [ee for ee in exemplars if ee in embs_dict]
    missing_exemplars = [ee for ee in exemplars if ee not in embs_dict]
    
    if missing_exemplars:
        print(f'\n**WARNING**: {len(missing_exemplars)} exemplars missing from {os.path.basename(embs_path)}: {missing_exemplars}')
        print(f'    Using only {len(available_exemplars)} available exemplars for this model')
    
    # Create the RDM using only available exemplars
    embs_arr = np.array([embs_dict[ee] for ee in available_exemplars])
    rdm = distance.pdist(embs_arr, metric='correlation')
    print(f'{os.path.basename(embs_path)} RDM squared has shape {distance.squareform(rdm).shape}')
    
    return rdm, embs_arr, embs_dict, available_exemplars


def subset_rdm(rdm_condensed, indices):
    """
    Subset a condensed RDM to only include specified exemplar indices.
    
    Parameters
    ----------
    rdm_condensed : array
        Condensed distance matrix from pdist
    indices : list of int
        Indices of exemplars to keep (in desired order)
    
    Returns
    -------
    rdm_subset : array
        Condensed RDM containing only the specified exemplars
    """
    rdm_square = distance.squareform(rdm_condensed)
    rdm_square_subset = rdm_square[np.ix_(indices, indices)]
    return distance.squareform(rdm_square_subset)

In [ ]:
# -------------------- Read exemplars list ( overlap in targets and objects)  -------------------- 
feats_legend_path = join(os.path.dirname(results_indir), 'feats_legend.p')
with open(feats_legend_path, 'rb') as f:
    feats_legend = pickle.load(f)


# define exemplars as the list of targets
exemplars = [i.split('_')[-1] for i in feats_legend if i.startswith('target_')]
exemplars = [i for i in exemplars if 'hand' not in i]

print('Num exemplars = ', len(exemplars))
print(exemplars)
print('')

# -------------------------------------- Read the RDMs --------------------------------------------
# Initialize dictionaries to store both RDMs and their exemplars
all_rdms = {}
all_rdm_exemplars = {}  # Track which exemplars each model has

# ********** semantic BGE **********
# Read embs
semantic_BGE_embs_path = join(rsa_model_rdm_dir, 'semantic/semantic_BGE_embs.p')
with open(semantic_BGE_embs_path, 'rb') as f:
    semantic_embs_dict_editedNames = pickle.load(f)

# Exception for semantic, we need to rename the embs since we had used alternate words to find the semantic embs
df_exemplar_names = pd.read_excel(join(rsa_model_rdm_dir, 'semantic/exemplar_names.xlsx'))
dict_rename_map = {row['exemplars_renamed'] : row['exemplars_in_targets_and_objects'] for idx,row in df_exemplar_names.iterrows()}

semantic_embs_dict = {}
for old_key,val in semantic_embs_dict_editedNames.items():
    new_key = dict_rename_map[old_key]
    semantic_embs_dict[new_key] = val

# Create the RDM
semantic_embs_arr = np.array([semantic_embs_dict[ee] for ee in exemplars])
semantic_rdm = distance.pdist(semantic_embs_arr, metric='correlation')
print('semantic RDM squared has shape ', distance.squareform(semantic_rdm).shape)

all_rdms['semantic_BGE'] = semantic_rdm
all_rdm_exemplars['semantic_BGE'] = exemplars  # semantic has all exemplars



# ********** Visual **********
visual_embs_path = join(rsa_model_rdm_dir, 'visual/visual_embs.p')
visual_rdm, visual_embs_arr, visual_embs_dict, visual_available_exemplars = read_embs_and_make_rdms(visual_embs_path, exemplars)

all_rdms['visual'] = visual_rdm
all_rdm_exemplars['visual'] = visual_available_exemplars



# ********** ROLE-AGNOSTIC Affordances **********
roleagnostic_affordance_embs_path = join(rsa_model_rdm_dir, 'affordance/roleagnostic_affordance_embs.p')
roleagnostic_affordance_rdm, roleagnostic_affordance_embs_arr, roleagnostic_affordance_embs_dict, roleagnostic_available_exemplars = read_embs_and_make_rdms(roleagnostic_affordance_embs_path, exemplars)

all_rdms['roleagnostic_affordance'] = roleagnostic_affordance_rdm
all_rdm_exemplars['roleagnostic_affordance'] = roleagnostic_available_exemplars




# ********** Hand linearcomb Affordances **********
hands_affordance_linearcomb_embs_path = join(rsa_model_rdm_dir, 'hands_affordance/hands_affordance_linearcomb_embs.p')
hands_affordance_linearcomb_rdm, hands_affordance_linearcomb_embs_arr, hands_affordance_linearcomb_embs_dict, hands_linearcomb_available_exemplars = read_embs_and_make_rdms(hands_affordance_linearcomb_embs_path, exemplars)

all_rdms['hands_linearcomb_affordance'] = hands_affordance_linearcomb_rdm
all_rdm_exemplars['hands_linearcomb_affordance'] = hands_linearcomb_available_exemplars

### Run the searchlight RSA

In [ ]:
# Read the mask that will determine where to do the searchlight (based on the encoding model group results)
with open(group_res_path, 'rb') as f:
    group_mask = pickle.load(f)

group_mask_img = group_mask['significance_mask']
del group_mask

In [ ]:
# Make output dir if it doesnt exist (/results inside the root dir)
outdir = os.path.dirname(outpath_template)
if not os.path.isdir(outdir):
    os.mkdir(outdir)

In [ ]:
conditions_of_interest = ['target', 'object']

In [ ]:
# Function to check if all conditions have been computed for this subject
def check_subject_conditions_complete(subj, conditions_of_interest, outpath_template):
    for cond in conditions_of_interest:
        outpath = outpath_template.format(subj, cond)
        if not os.path.exists(outpath):
            return False
    return True

for subj in tqdm(subjects, desc='Subjects'):
    print('Subj =', subj)
    
    if check_subject_conditions_complete(subj, conditions_of_interest, outpath_template):
            print(f"All conditions already computed for {subj}. Skipping.")
            continue
    
    else:    
        # Read ridge results
        ridge_results_path =  join(results_indir, subj,  f'{subj}_ridge_results.p')
        with open(ridge_results_path, 'rb') as f:
            banded_ridge_results = pickle.load(f)
        
        # ------------------------ mask needed to reconstruct volumetric data ---------------
        subj_brainmask_img = image.load_img(path_config.get_brainmask_path(subj))
        mask_modelled_voxels_img = unmask(banded_ridge_results['mask_modelled_voxels'], subj_brainmask_img) # reconstruct the 3d mask for use later, and for viz here


        # --------------------------- Coefficients ---------------------------------------------
        # get the coefs
        subj_coeffs = banded_ridge_reconstruct_fir_weights(banded_ridge_results, verbose=False)
        # convert into a dict where the keys are the features names and the values are the corresponding coeff_img (masked  for significance)
        feats_legend = banded_ridge_results['feats_legend']

        # convert back to img
        init_coeffs_img = unmask(subj_coeffs, mask_modelled_voxels_img) # shape is now (96, 114, 96, 169) -> (x,y,x, num_feats)

        # Mask out non significant voxels using the group results mask
        init_coeffs_img = replace_ns_with_nan(init_coeffs_img, group_mask_img)

        
        
        for cond in conditions_of_interest:
            print('condition =', cond)
            
            outpath = outpath_template.format(subj, cond)

            # If already done, skip
            if os.path.exists(outpath):
                print(f"Result found: {outpath}\n Skipping {subj}-{cond}.")
            
            else:
                # ======================== Create the neural RDMs using searchlight ===========================
                # indices of the features of interest
                idx_feats_of_interest = [feats_legend.index(f'{cond}_{i}') for i in exemplars]

                # subset coefficient images by the feats of interest
                coeffs_imgs, labels = [],[]
                for ii in idx_feats_of_interest:
                    coeffs_imgs.append(init_coeffs_img.slicer[:,:,:,ii])
                    labels .append(feats_legend[ii])

                # Create searchlight centers and neighbors, based on group significance mask
                group_mask_array = group_mask_img.get_fdata()
                centers, neighbors = get_volume_searchlight(group_mask_array, radius=radius, threshold=threshold)

                # reshape the coefficients into (n_observations, 96, 114, 96)
                data = image.concat_imgs(coeffs_imgs) # shape is (96, 114, 96, n_observations)  where n_observations is n_exemplars
                data = data.get_fdata()
                data = np.transpose(data, (3,0,1,2))  # now shape is (n_observations, 96, 114, 96)

                # reshape data so we have n_observastions x n_voxels
                data_2d = data.reshape([data.shape[0], -1])
                # Replace NaN with 0 to avoid crashes in evaluate_models_searchlight(); group mask is re-applied after scoring
                data_2d = np.nan_to_num(data_2d)

                # Get the neural RDMs
                SL_RDM = get_searchlight_RDMs(data_2d, centers, neighbors, labels, method='correlation')

                # ======================== Run RSA on neural RDMs ===========================
                models_results_dict = {} # output dict for the current condition: one key,value per model tested

                for model_name, model_rdm_condensed in all_rdms.items():
                    print('Testing model RDM =', model_name)

                    # Get available exemplars for this model
                    model_exemplars = all_rdm_exemplars[model_name]

                    # Check if we need to subset. Should only be for hands_linearcomb where we are missing some exemplars
                    if len(model_exemplars) < len(exemplars):
                        print(f"  {model_name} missing some exempalrs. Subsetting to {len(model_exemplars)} available exemplars")
                        # Find indices of available exemplars
                        idx_available_exemplars_neural = [i for i, ee in enumerate(exemplars) if ee in model_exemplars] # Get the positions of shared exemplars in the NEURAL DATA list (original exemplars list). This tells us which rows to extract from data_
                        idx_available_exemplars_model = [model_exemplars.index(exemplars[i]) for i in idx_available_exemplars_neural] # Get the positions of the same shared exemplars in the MODEL list (model_exemplars). This tells us which elements to extract from the model RDM
                        
                        # Subset the neural data
                        data_2d_subset = data_2d[idx_available_exemplars_neural, :] # subset along the exemplars/observations dimension
                        labels_subset = [labels[i] for i in idx_available_exemplars_neural]
                        
                        # Get new searchlight RDMs with subset
                        SL_RDM_subset = get_searchlight_RDMs(data_2d_subset, centers, neighbors, 
                                                            labels_subset, method='correlation')
                        
                        # Subset model RDM
                        model_rdm_subset = subset_rdm(model_rdm_condensed, idx_available_exemplars_model)
                        my_model = ModelFixed(model_name, model_rdm_subset)
                        
                        # Evaluate (using SL_RDM_subset)
                        searchlight_results = evaluate_models_searchlight(SL_RDM_subset, my_model,  # list of len=number of centers, each element is a rsatoolbox.inference.result.Result object
                                                                eval_fixed, method='spearman', 
                                                                n_jobs=n_jobs)
                        # to reconstruct the volume later
                        voxel_index = list(SL_RDM_subset.rdm_descriptors['voxel_index']) # using SL_RDM_subset

                    else:
                        # No subsetting needed, simple implementation
                        my_model = ModelFixed(model_name, model_rdm_condensed)
                        searchlight_results = evaluate_models_searchlight(SL_RDM, my_model,  # list of len=number of centers, each element is a rsatoolbox.inference.result.Result object
                                                                eval_fixed, method='spearman', 
                                                                n_jobs=n_jobs)
                        # to reconstruct the volume later
                        voxel_index = list(SL_RDM.rdm_descriptors['voxel_index'])  # using SL_RDM

                    # get the scores (as list)
                    searchlight_scores = [float(e.evaluations) for e in searchlight_results]  # list of len=number of centers, each element now a score (float)


                    # Convert the result into an img. Pulled from the tutorial.
                    x, y, z = group_mask_img.shape # here using group_mask_img
                    results_img = np.full((x * y * z,), np.nan) # create an empty array to house the results
                    results_img[voxel_index] = searchlight_scores
                    results_img = results_img.reshape([x, y, z])
                    results_img = image.new_img_like(group_mask_img, results_img)
                    results_img = replace_ns_with_nan(results_img, group_mask_img) # re-apply the group mask.

                    # Add to the results dict
                    models_results_dict[model_name] = results_img


                # Save results to file. (one per subject-condition pair)
                with open(outpath, 'wb') as f:
                    pickle.dump(models_results_dict, f)

                print(f'Done. Results saved to {outpath}')


print('All Done.')

### Group-stats

- For each condition (target, object)
    - For each model RDM: 
        - Run group_level_1sample_ttest() checking for RSA score (correlation) > 0
    - Plot the winner takes all map

In [ ]:
choose_RDMS_to_test = ['semantic_BGE', 'visual', 'roleagnostic_affordance', 'hands_linearcomb_affordance']

In [ ]:
import numpy as np
from nilearn.image import new_img_like

def fisher_z_img(r_img):
    """Apply Fisher z-transform voxelwise using arctanh."""
    r = r_img.get_fdata()

    # Clip values to avoid ±1 issues (arctanh undefined at ±1)
    r = np.clip(r, -0.999999, 0.999999)

    z = np.arctanh(r)
    return new_img_like(r_img, z)

def fisher_r_img(z_img):
    """Convert Fisher z values back to r."""
    z = z_img.get_fdata()
    r = np.tanh(z)
    return new_img_like(z_img, r)


In [ ]:
from matplotlib.patches import Patch

# summary plots outpath
wta_outdir = 'wta_plots'
if not os.path.isdir(wta_outdir):
    os.mkdir(wta_outdir)


# read template mni brain
anat_img = image.load_img(anat_img_path)

all_group_results = {}
all_raw_group_res_outputs = {}  # Stores raw group-level t-test outputs; keys are [condition][model_name]



# ============================================
# CREATE CONSISTENT COLOR MAPPING
# ============================================
all_model_names = list(all_rdms.keys())
n_models_total = len(all_model_names)

# Create a consistent colormap for ALL models
cmap_full = mpl.colormaps["rainbow"].resampled(n_models_total)

# Map each model to a specific color index and color
model_to_idx = {name: i for i, name in enumerate(all_model_names)}
model_to_color = {name: cmap_full(i) for i, name in enumerate(all_model_names)}

# Track which models appear in at least one condition
models_with_sig_voxels = set()

##############################################
# WINNER-TAKES-ALL ACROSS MODELS (per condition)
##############################################

for cond in conditions_of_interest:
    print("\n===============================")
    print(f"Condition: {cond}")
    print("===============================\n")

    # ----------------------------------------------
    # 1) Load results for all models & all subjects
    # ----------------------------------------------
    condition_results = {model_name: [] for model_name in all_rdms.keys()}

    for subj in subjects:
        results_path = outpath_template.format(subj, cond)
        with open(results_path, 'rb') as f:
            results = pickle.load(f)
        
        for model_name in all_rdms.keys():
            # Fisher z-transform (subject-level)
            r_img = results[model_name]
            z_img = fisher_z_img(r_img)

            # Store z-values for group-level statistics
            condition_results[model_name].append(z_img)

    # ----------------------------------------------
    # 2) Group-level stats per model
    # ----------------------------------------------
    group_avg_imgs_sig = []
    model_names_ordered = []
    sig_masks = []

    for model_name in choose_RDMS_to_test:
        print(f"Testing model: {model_name}")

        model_results_list = condition_results[model_name]

        group_res = group_level_1sample_ttest(
            model_results_list,
            popmean=0,
            mask_img=group_mask_img,
            fwhm=6,
            alpha=0.05,
            height_control='fdr',
            cluster_threshold=10,
            two_sided=False,
            verbose=False
        )

        # Append to the subject-level results dict for later use if needed
        all_raw_group_res_outputs.setdefault(cond, {})[model_name] = group_res

        sig_mask_img = group_res['significance_mask']
        n_sig_voxels = np.sum(sig_mask_img.get_fdata() != 0)

        if n_sig_voxels > 0:
            print(f"  -> Significant voxels found: {n_sig_voxels}")

            avg_img = average_list_of_imgs(model_results_list)
            avg_img_sig = replace_ns_with_nan(avg_img, sig_mask_img)

            group_avg_imgs_sig.append(avg_img_sig) 
            model_names_ordered.append(model_name)
            sig_masks.append(sig_mask_img)
            
            # Track this model as having significant voxels
            models_with_sig_voxels.add(model_name)
        else:
            print(f"  ✗ No significant voxels.")

    if len(group_avg_imgs_sig) == 0:
        print(f"\n No models showed significant voxels for condition {cond}. Skipping.")
        continue

    print("\n--- Computing winner-takes-all across models ---\n")

    # -----------------------------------------------------------
    # 3) Append result
    # -----------------------------------------------------------
    all_group_results[cond] = {
        'group_avg_imgs_sig': group_avg_imgs_sig,
        'model_names_ordered': model_names_ordered,
        # 'wta_img': wta_img
    }


### Plot the result map for each condition and model 
3 plots are created for each conditio-model pair
- flatmap
- mosaic plot
- volumetric plot of the peak correlation

In [ ]:
from get_topN_cluster_coordinates import get_topN_cluster_coordinates

In [ ]:
model_specific_outdir = 'model_specific_plots'
if not os.path.isdir(model_specific_outdir):
    os.mkdir(model_specific_outdir)

for cond in conditions_of_interest:
    model_names_ordered = all_group_results[cond]['model_names_ordered'] # to make sure we grab the right data
    
    for model_idx, model_name in enumerate(model_names_ordered):     
        print(cond, model_name)
        searchlight_img = all_group_results[cond]['group_avg_imgs_sig'][model_idx]

        # ----------------------- flatmap ---------------------------
        flatmap, _ = pycortex_plot_flatmap(
                            searchlight_img, template_subj,
                            cortex_Volume_kwargs=dict(vmin=0, cmap='YlOrRd'),
                            cbar_label= None,
                            quickflat_kwargs=quickflat_defaults
                            
                            
                            # title=f"{cond} - {model_name}"
                        )
        flatmap.savefig(join(model_specific_outdir, f"flatmap_{cond}_{model_name}.png"), bbox_inches='tight', dpi=300)
        # -------------------- Mosaic plot ---------------------------
        fig1, ax1 = plt.subplots(figsize=(8,4)) 
        ax1.set_axis_off() # remove all ax1 borders/axis

        mosaic_plot = plot_stat_map(
            searchlight_img,
            bg_img=anat_img,
            cmap='YlOrRd',
            symmetric_cbar=False,
            threshold=0,
            vmin=0,
            display_mode='mosaic',
            cut_coords=4,
            black_bg=False,
            figure=fig1
            # title=f"{cond} - {model_name}"
        )
        mosaic_plot._cbar.set_label("Spearman's ρ", fontsize=11, fontweight='bold')
        cleanup_cbar(mosaic_plot)

        fig1.savefig(join(model_specific_outdir, f"mosaic_{cond}_{model_name}.png"), bbox_inches='tight', dpi=300)

        # -------------------- Peak cluster --------------------------
        
        fig2, ax2 = plt.subplots(figsize=(8,4)) 
        ax2.set_axis_off() # remove all ax1 borders/axis

        clusters = get_topN_cluster_coordinates(
            searchlight_img,
            threshold=0,
            top_n=1,
            direction='positive',
            verbose=True)
        top_clu = clusters['positive'][0]

        display = plot_img(
                                searchlight_img,
                                bg_img=anat_img,
                                threshold=0,
                                cut_coords=top_clu,
                                colorbar=True,
                                cmap = 'YlOrRd',
                                vmin = 0,
                                black_bg=False,
                                draw_cross=False,
                                figure=fig2
                                # title=f"Peak: {cond} - {model_name}"
                        )
        display._cbar.set_label("Spearman's ρ", fontsize=11, fontweight='bold')
        cleanup_cbar(display)
        custom_crosshair(display)


        # --- make edits to the cbar ---
        # Shrink the colorbar
        cbar_ax = display._cbar.ax
        pos = cbar_ax.get_position()
        new_height = pos.height * 0.5  # Make it 50% of original height
        cbar_ax.set_position([pos.x0, pos.y0 + (pos.height - new_height)/2, pos.width, new_height])


        fig2.savefig(join(model_specific_outdir, f"peak_{cond}_{model_name}.png"), bbox_inches='tight', dpi=300)
